In [ ]:
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import sys

In [ ]:
sys.path.append('../src/nlr_ghe_screen/clustering/')
from execute_clustering import (
    aggregate_building_data,
    aggregate_load_data,
    execute_milp
)
from seed_clustering import (
    find_seed_centroids_single
)
from cluster_construction import (
    find_and_filter_nearby_buildings,
)
from distance_metrics import (
    euclidian_distance_to_centroid,
)

In [ ]:
from plotting_functions import (
    plot_timeseries_loads
)

In [ ]:
# Get the bounding box of the area that is to be investigated
TEST_BBOX = (47.58, -122.36, 47.61, -122.31) # xmin, ymin, xmax, ymax
south, west, north, east = TEST_BBOX

In [ ]:
# Get the pre-queried GHE / waste heat source locations
ghe_locations = gpd.read_file("../src/exports/ghe_locations.geojson")

# Crop the locations to the bounding box
geom_box = box(west, south, east, north)
cropped_ghes = gpd.clip(ghe_locations, geom_box)
cropped_ghes = cropped_ghes.to_crs(cropped_ghes.estimate_utm_crs())

In [ ]:
# Get building data from Overture Maps and ORNL USA Structures
building_data = aggregate_building_data(
    bbox=TEST_BBOX, 
    crs=cropped_ghes.crs
)

In [ ]:
# Create seed centroids from each GHE / waste heat source location
labeled_ghes, seed_centroids = find_seed_centroids_single(
    ghe_locations=cropped_ghes,
)

# For each of the centroid
# gather all buildings within a certain road network radiuss
nearby_buildings = find_and_filter_nearby_buildings(
    seed_centroids=seed_centroids,
    building_data=building_data,
    max_distance=500,
)

In [ ]:
# Get all buildings near the selected cluster
# and the centroid associated with that cluster
seed_idx = 250
building_set = nearby_buildings[nearby_buildings['cluster'] == seed_idx].copy()
cluster_seed_centroid = seed_centroids.loc[seed_centroids['cluster'] == seed_idx, 'geometry'].item()

# Find the distance between each of the buildings and the seed centroid
building_set['centroid_distance'] = euclidian_distance_to_centroid(
    buildings_df=building_set,
    centroid=cluster_seed_centroid,
    max_distance=500
)

In [ ]:
# Gather load profiles for each building in the selected cluster
building_loads = aggregate_load_data(
    building_data=building_set
)

In [ ]:
# Isolate the buildings whose profiles were able to be retrieved 
building_ids = [b.attrs['bldg_id'] for b in building_loads]
b = building_set.loc[building_ids].copy()


In [ ]:
# Execute the MILP 
results = []
unique = set()
for wp, wi in [(i, 1 - i) for i in np.arange(0, 1, 0.1)]:
    milp_result = execute_milp(
        building_data=b,
        building_loads=building_loads,
        seed_idx=0,
        # k_min=3,
        weight_peak=wp,
        weight_net=wi
    )

    if tuple(milp_result['selected']) not in unique:
        results.append(milp_result)
        unique.add(tuple(milp_result['selected']))

In [ ]:
plot_timeseries_loads(
    [res['norm_curve'] for res in results],
    ['' for _ in range(len(results))],
    "Normalized Final Cluster Loads",
    "Time (day)",
    "Thermal Demand"
)

In [ ]:
result = results[0]
selected = result['selected']
final_cluster = b.iloc[selected]

In [ ]:
m = labeled_ghes[labeled_ghes['cluster'] == seed_idx].explore(color='green')
seed_centroids[seed_centroids['cluster'] == seed_idx].explore(m=m, color='red')
building_set.explore(m=m, color='blue')

In [ ]:
m = labeled_ghes[labeled_ghes['cluster'] == seed_idx].explore(color='green')
seed_centroids[seed_centroids['cluster'] == seed_idx].explore(m=m, color='red')
final_cluster.explore(m=m, color='blue')